**Question** Does performance survive scenarios that are not represented in the ordinary training split?
**Method** Use configuration-driven scenario metadata and the repository multi-topology artifact; never fit or tune on these scenario rows.
**Takeaway** Generalization claims are conditional and are reported with not-applicable markers when a fixed-width model cannot accept a new topology.

In [ ]:
from pathlib import Path
import json, pandas as pd
ROOT=Path.cwd()
scenarios=['unseen topology: 4/6/7 nodes','leave-one-fault-type-out','unseen fault node','step ramp','exponential ramp','2-3x noise','20% missing metrics','compound faults']
records=[]
for scenario in scenarios:
    records.append({'scenario':scenario,'training_policy':'train only on in-distribution rows','flat_model_status':'not applicable: fixed 5-node width' if 'topology' in scenario else 'evaluate without retuning','score_source':'computed runner output required'})
display(pd.DataFrame(records))

### What this tells us
The scenario matrix records the evaluation policy before scores are inspected. A changed topology is not silently passed through a fixed-width flat model; it is marked not applicable unless a per-node representation is implemented.

In [ ]:
import torch
path=ROOT/'dataset/processed/multi_topology_tensors.pt'
splits_path=ROOT/'dataset/processed/multi_topology_splits.json'
if path.exists() and splits_path.exists():
    tensors=torch.load(path,map_location='cpu',weights_only=False); splits=json.loads(splits_path.read_text())
    print({k: len(v) if isinstance(v,list) else v for k,v in splits.items()})
    print('tensor shapes:', {k:tuple(v.shape) for k,v in tensors.items() if hasattr(v,'shape')})
else:
    print('Cross-topology artifact is absent; scenario evaluation is skipped honestly.')

### What this tells us
The repository artifact, when present, gives a true unseen-topology holdout with node masks. If it is absent in a clean Kaggle run, the notebook reports a skip rather than fabricating degradation values.

In [ ]:
result_path=ROOT/'notebooks/results/unseen_scenario_results.json'
if result_path.exists():
    unseen=pd.DataFrame(json.loads(result_path.read_text()))
    display(unseen)
else:
    print('No unseen-scenario result artifact is committed. Run the scenario script after extending the generator config.')

### What this tells us
This result gate prevents the notebook from presenting an imagined degradation curve. The generator extension remains backward compatible by design: scenario definitions belong in configuration, and unseen rows must be created before this table can contain measurements.

## External-data note
An external Kaggle distributed-systems dataset is evaluated only if a local file is explicitly supplied. No external file is present in this repository, so no cross-dataset score is claimed.